# AI Agent & Tool Calling 실습 노트북

이 노트북은 **LangChain 기반 AI Agent**와 **Tool Calling** 개념을 실습합니다.

---
**주요 실습 내용**
- `@tool` 데코레이터로 커스텀 도구(Tool) 정의
- `llm.bind_tools()` 로 LLM에 도구 바인딩 (버전 호환)
- Tool 호출 루프를 직접 구현하여 Agent 동작 원리 이해
- 3번 노트북의 FAISS 벡터스토어를 **RAG Tool**로 등록
- 대화 히스토리를 유지하는 대화형 Agent 실습

---
**AI Agent 동작 흐름**
```
사용자 질문
  → LLM이 필요한 Tool 판단 및 tool_calls 생성
  → Tool 실행 → 결과를 ToolMessage로 LLM에 전달
  → tool_calls 없을 때까지 반복
  → 최종 답변 반환
```


## 1. 패키지 설치 및 환경 설정

In [ ]:
%pip install -r ../requirements.txt

from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key, "OPENAI_API_KEY 환경변수가 필요합니다."

from langchain_openai import ChatOpenAI
import langchain

llm = ChatOpenAI(model="gpt-4o", openai_api_key=openai_api_key, temperature=0)
print(f"LangChain 버전: {langchain.__version__}")
print("환경 설정 완료!")

## 2. Tool 정의

`@tool` 데코레이터를 붙이면 함수의 **이름**과 **docstring**이 LLM에 전달됩니다.
LLM은 docstring을 보고 언제 이 도구를 쓸지 판단합니다.

- `get_weather`: 도시 날씨 조회 (Mock 데이터)
- `calculate`: 수식 계산

In [ ]:
from langchain_core.tools import tool
import math


@tool
def get_weather(city: str) -> str:
    """주어진 도시의 현재 날씨 정보를 반환합니다. 날씨를 물어볼 때 사용하세요."""
    weather_data = {
        "서울": "맑음, 기온 22°C, 습도 60%",
        "부산": "흐림, 기온 25°C, 습도 75%",
        "제주": "비, 기온 20°C, 습도 85%",
        "대전": "맑음, 기온 21°C, 습도 55%",
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 표현식을 계산합니다. 사칙연산, sqrt, pow, pi, log 등을 지원합니다. 예: '2 + 3 * 4', 'sqrt(16)', 'pi * 5**2'"""
    try:
        safe_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}}, safe_names)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# 도구 단독 테스트
print("[get_weather]", get_weather.invoke("서울"))
print("[calculate] ", calculate.invoke("sqrt(144) + 2**10"))

## 3. llm.bind_tools() 로 Tool Calling Agent 구성

**`llm.bind_tools(tools)`** 는 LangChain 버전에 관계없이 사용 가능한 Tool Calling 방식입니다.

- `bind_tools()`: LLM에 사용 가능한 도구 목록을 등록
- `response.tool_calls`: LLM이 어떤 도구를 어떤 인자로 호출할지 결정한 결과
- `ToolMessage`: 도구 실행 결과를 LLM에 전달하는 메시지 타입

아래에서 **Agent 루프**를 직접 구현해 동작 원리를 확인합니다.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage


def run_agent(user_input: str, tools: list, system: str = None, chat_history: list = None, verbose: bool = True) -> str:
    """
    Tool Calling Agent 루프.
    LLM이 tool_calls를 반환하지 않을 때까지 Tool 실행을 반복합니다.
    """
    llm_with_tools = llm.bind_tools(tools)
    tools_map = {t.name: t for t in tools}

    # 메시지 초기화
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    if chat_history:
        messages.extend(chat_history)
    messages.append(HumanMessage(content=user_input))

    step = 1
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # tool_calls 가 없으면 최종 답변
        if not response.tool_calls:
            break

        # Tool 실행
        for tc in response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            tool_result = tools_map[tool_name].invoke(tool_args)

            if verbose:
                print(f"  [Step {step}] Tool: {tool_name}({tool_args})")
                print(f"           → {tool_result}")

            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tc["id"]
            ))
            step += 1

    return response.content


# 기본 도구 목록
basic_tools = [get_weather, calculate]

# 테스트: 두 가지 도구를 한 번에 사용
print("질문: 서울 날씨 알려줘. 그리고 반지름 7인 원의 넓이(pi*r^2)는?\n")
answer = run_agent(
    "서울 날씨 알려줘. 그리고 반지름 7인 원의 넓이(pi*r^2)는?",
    basic_tools
)
print(f"\n최종 답변: {answer}")

## 4. RAG Tool 추가 (FAISS 벡터스토어 연동)

3번 노트북에서 저장한 FAISS 벡터스토어를 **검색 도구**로 등록합니다.
Agent는 UiPath 관련 질문이 들어오면 자동으로 이 도구를 호출합니다.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=openai_api_key)
persist_dir = "../faiss_uipath"

assert os.path.exists(persist_dir), "⚠️ FAISS 인덱스가 없습니다. 3번 노트북을 먼저 실행해 주세요."

vectorstore = FAISS.load_local(
    folder_path=persist_dir,
    embeddings=embedding,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("FAISS 벡터스토어 로드 완료!")


@tool
def search_uipath_docs(query: str) -> str:
    """UiPath 문서에서 관련 정보를 검색합니다. UiPath, RPA, 테스트 자동화, Studio Pro, Orchestrator, Test Manager에 관한 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾을 수 없습니다."
    return "\n\n".join([f"[문서 {i+1}]\n{doc.page_content}" for i, doc in enumerate(docs)])


# RAG Tool 단독 테스트
print("\n[RAG Tool 테스트]")
print(search_uipath_docs.invoke("UiPath Studio Pro 기능")[:400], "...")

## 5. Multi-tool Agent 실습

날씨, 계산기, RAG 검색 도구를 모두 등록합니다.
Agent는 질문의 의도에 따라 적합한 도구를 **자동 선택**합니다.

In [ ]:
all_tools = [get_weather, calculate, search_uipath_docs]

SYSTEM = "당신은 유능한 AI 비서입니다. 필요한 경우 도구를 사용해 사용자를 도와주세요. 한국어로 답변하세요."

# 테스트 1: RAG 도구가 필요한 질문
print("=" * 60)
print("[질문 1] UiPath Orchestrator 역할")
print()
ans1 = run_agent("UiPath Orchestrator는 어떤 역할을 해?", all_tools, system=SYSTEM)
print(f"\n최종 답변: {ans1}")

In [ ]:
# 테스트 2: 여러 도구를 조합해야 하는 복합 질문
print("=" * 60)
print("[질문 2] 복합 질문 (RAG + 날씨)")
print()
ans2 = run_agent(
    "UiPath Test Manager가 뭔지 간단히 설명해줘. 그리고 제주 날씨도 알려줘.",
    all_tools,
    system=SYSTEM
)
print(f"\n최종 답변: {ans2}")

## 6. 대화 히스토리를 유지하는 Agent

이전 대화를 `chat_history` 리스트에 쌓아서 매 턴마다 `run_agent`에 전달합니다.
Agent는 이전 맥락을 참고하여 이어서 답변합니다.

In [ ]:
from langchain_core.messages import AIMessage

# 대화 히스토리 초기화
chat_history = []


def chat(user_input: str):
    """히스토리를 유지하며 Agent와 대화"""
    answer = run_agent(
        user_input,
        tools=all_tools,
        system=SYSTEM,
        chat_history=chat_history,
        verbose=False  # 중간 과정 생략
    )
    # 히스토리에 사용자 메시지와 AI 답변 추가
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer))

    print(f"사용자: {user_input}")
    print(f"Agent : {answer}")
    print("-" * 60)


# 연속 대화 테스트
chat("UiPath Test Suite가 뭐야?")                        # RAG 도구 사용
chat("방금 설명한 제품에서 Studio Pro는 어떤 역할이야?")  # 이전 답변 맥락 참조
chat("서울 날씨도 알려줘")                               # 날씨 도구로 전환

## 정리

| 개념 | 설명 |
|------|------|
| `@tool` | 함수를 LLM이 호출 가능한 도구로 변환. docstring이 LLM에 전달됨 |
| `llm.bind_tools(tools)` | LLM에 사용할 도구 목록을 등록. |
| `response.tool_calls` | LLM이 호출하길 원하는 도구명과 인자 목록 |
| `ToolMessage` | 도구 실행 결과를 LLM에 전달하는 메시지 타입 |
| `chat_history` | 이전 대화 메시지를 누적하여 맥락 유지 |


**다음 단계 아이디어:**
- 외부 API(실제 날씨, 웹 검색 등) 연동 도구 추가
